# Testing the Agentic Knowledge Engineering Tool
This notebook sets up the `Orchestrator` to test the Distillation, Designer, and Validation loops with `instructor` and `openai` against your configured LLM (e.g. Vertex AI or local Ollama).


In [1]:
import sys
import os
# Allow imports from local directory if running inside the ontology folder
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from core.models import DocumentSource
from pipeline.orchestrator import Orchestrator


## Define Test Documents
Here we define the input raw texts that need to be distilled into a Knowledge Graph and synthesized into a strict Pydantic schema.


In [2]:
doc1 = DocumentSource(
    id="doc_001",
    text_content="""
    Patient Assessment - Oct 4, 2023:
    Jordan exhibited highly repetitive physical motions, specifically hand-flapping, 
    when the classroom environment became too loud. He did not engage in verbal communication for 45 minutes.
    """
)

doc2 = DocumentSource(
    id="doc_002",
    text_content="""
    In-home Observation - Oct 5, 2023:
    During dinner, Jordan successfully maintained eye contact with his sibling when asking for water.
    However, when transitioned to bedtime, severe physical agitation and distress was observed.
    """
)

docs = [doc1, doc2]


## Initialize Orchestrator and Run Pipeline
We point the Orchestrator to the configured LLM endpoint using generic environment variables (`LLM_BASE_URL`). Note: ensure your endpoint is running or correct credentials are provided."


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from pipeline.orchestrator import Orchestrator

import asyncio

orchestrator = Orchestrator(
    max_concurrency=4,
    hallucination_filter=True,
    ontology_depth=None,
    strict_typing=True,
    verbose=True,
    generalize_latent_space=True,
    generalize_structural_roles=True,
    generalize_taxonomic_lifting=True,
    generalize_seeded_schemas=True,
    save_stage_outputs=True
)

async def main():
    try:
        final_schema = await orchestrator.run_pipeline(docs)
        return final_schema
    except Exception as e:
        print(f"Pipeline Error: {e}")
        print("Is your LLM connection running locally, or did you export LLM_BASE_URL?")
        return None

final_schema = await main()


[*] Orchestrator initialized. Model: mistral-nemo:12b-instruct-2407-q4_K_M | Base URL: http://localhost:11434/v1




################ PHASE 1. LATENT DISCOVERY CLUSTERING ################

====== STAGE 0. DISCOVERY LOOP ======
[*] Executing schema-less triple extraction over 2 documents...


[TIMER] ExplorerEngine.extract_raw_triples for doc 'doc_001' took: 6.50s
[CHECKPOINT] Document 'doc_001' done scanning for triples.


[TIMER] ExplorerEngine.extract_raw_triples for doc 'doc_002' took: 10.22s
[CHECKPOINT] Document 'doc_002' done scanning for triples.


[+] Extracted 6 raw topological triples.
[SAVE] Stage output written to stage_outputs/1_raw_triples.json
[*] Running algorithmic Louvain community detection natively routing the graph...
[VERBOSE] K-Core Pruning: Sequestered 4 leaf noise nodes. Core graph contains 2 dense nodes.
[VERBOSE] Auto-Resolution Sweeper: Target max bounding limit = 0 communities.
[VERBOSE] Structural Jaccard Merger: Cross-evaluating 1 atomic communities for logical isomorphism.
[VERBOSE] Topology Refinement: Jaccard overlay compressed 1 raw logic atoms solidly into exactly 1 unified master schemas.
[TIMER] Louvain graph isolation mapped in 0.00s
[+] Detected 1 distinct logic clusters.
[*] Hardening clusters via Canonical Pydantic Schema Translation...
  -> Validating Cluster 1 (6 nodes)...


...


====== STAGE 0.5 GENERALIZATION (PHASE 3) ======
[*] Generalizer: Analyzing Structural Motifs & Roles (Code-only Mathematical heuristic)...
[*] Generalizer: Mapping to Seeded Roots (6 anchors)...


[SAVE] Stage output written to stage_outputs/2_hardened_clusters.json
[*] Generating dynamic constraint-based Python models...
[+] Successfully exported rigid Pydantic models to generated_schemas/models.py natively.
\n[VERBOSE] Explicit Hardened Pydantic Blueprint Configuration:
{
  "$defs": {
    "AgentModel": {
      "description": "Discovered Class: Agent.\\nNEGATIVE CONSTRAINTS:\\n- never_does_not_walk\\n- )-> []: The absence of a specific target in this edge suggests that Jordan can exhibit distress or physical agitation at other times and circumstances, but it's explicitly stated that he maintained eye contact during dinner on Oct 5th while asking for water.\\n- never_engages_in_except_verbal_communication\\n- This is deduced because no other types of communication are mentioned in the cluster.\\n- never_exhibited_external_loud_noise_excuses\\n- although it's explicitly stated that Jordan exhibited hand-flapping due to the classroom becoming loud, there's no evidence suggesting t

[TIMER] Constrained Extraction for doc 'doc_002' took: 5.60s
[CHECKPOINT] Document 'doc_002' done constrained mapping.

[VERBOSE] Constrained Payload:
{
  "agent": {
    "exhibited_physical_motions": "*aggitation*",
    "exhibited_hand_flapping": "N/A",
    "caused_by_classroom_loudness": "N/A",
    "did_not_engage_in_verbal_communication": "N/A",
    "maintained_eye_contact_in_specific_context": "N/A",
    "exhibited_distress_at_bedtime": "1.0 (Observed directly at 'bedtime')"
  },
  "reasoning": "**During** dinner, Jordan **successfully maintained** eye contact with his sibling when asking for water."
}


[TIMER] Constrained Extraction for doc 'doc_001' took: 11.14s
[CHECKPOINT] Document 'doc_001' done constrained mapping.

[VERBOSE] Constrained Payload:
{
  "agent": {
    "exhibited_physical_motions": "Jordan",
    "exhibited_hand_flapping": "Jordan",
    "caused_by_classroom_loudness": "the classroom environment became too loud",
    "did_not_engage_in_verbal_communication": "He did not engage in verbal communication for 45 minutes",
    "maintained_eye_contact_in_specific_context": "Jordan",
    "exhibited_distress_at_bedtime": "He exhibited distress at bedtime."
  },
  "reasoning": "His mother reported that he typically maintains eye contact in specific contexts, and recently he has been exhibiting distress at bedtime."
}


\n[+] SUCCESS: Bypassed Legacy Ontologist entirely. The pipeline output is structurally locked to mathematical heuristics!
[SAVE] Stage output written to stage_outputs/4_final_distilled_outputs.json


## Synthesized Schema Output
If the validation passes strictly with zero false positives, we can now view the dynamic schema produced.


In [4]:
if final_schema:
    print("Final Extracted Data:\n")
    import json
    for i, item in enumerate(final_schema):
        print(f"Document {i+1}:")
        print(item.model_dump_json(indent=2))
        
    print("\n\nDynamic Schema Used:\n")
    print(json.dumps(final_schema[0].model_json_schema(), indent=2))


Final Extracted Data:

Document 1:
{
  "agent": {
    "exhibited_physical_motions": "Jordan",
    "exhibited_hand_flapping": "Jordan",
    "caused_by_classroom_loudness": "the classroom environment became too loud",
    "did_not_engage_in_verbal_communication": "He did not engage in verbal communication for 45 minutes",
    "maintained_eye_contact_in_specific_context": "Jordan",
    "exhibited_distress_at_bedtime": "He exhibited distress at bedtime."
  },
  "reasoning": "His mother reported that he typically maintains eye contact in specific contexts, and recently he has been exhibiting distress at bedtime."
}
Document 2:
{
  "agent": {
    "exhibited_physical_motions": "*aggitation*",
    "exhibited_hand_flapping": "N/A",
    "caused_by_classroom_loudness": "N/A",
    "did_not_engage_in_verbal_communication": "N/A",
    "maintained_eye_contact_in_specific_context": "N/A",
    "exhibited_distress_at_bedtime": "1.0 (Observed directly at 'bedtime')"
  },
  "reasoning": "**During** dinner